In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from PIL import Image


# ============================================================
# PHASE 2 — DATASET AND DATALOADERS
# (same as before — no changes here)
# ============================================================

DATASET_PATH = Path("/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset/Combined_Dataset/IND_and_NEP")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.3, hue=0.1),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class_names = sorted([
    f.name for f in DATASET_PATH.iterdir() if f.is_dir()
])
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

all_samples = []
for class_name in class_names:
    label = class_to_idx[class_name]
    for img_file in (DATASET_PATH / class_name).iterdir():
        if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            all_samples.append((img_file, label))

random.seed(42)
random.shuffle(all_samples)

total     = len(all_samples)
train_end = int(0.70 * total)
val_end   = int(0.85 * total)

train_samples = all_samples[:train_end]
val_samples   = all_samples[train_end:val_end]
test_samples  = all_samples[val_end:]

class AQIDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label

train_dataset = AQIDataset(train_samples, train_transform)
val_dataset   = AQIDataset(val_samples,   val_test_transform)
test_dataset  = AQIDataset(test_samples,  val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32,
                          shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Classes: {class_names}")


# ============================================================
# PHASE 3 — RESNET50 BACKBONE + PATCHIFY
#
# WHAT CHANGED FROM BEFORE:
# StridedConvBlock is completely removed.
# ResNet50 replaces it as the feature extractor.
#
# WHY RESNET50?
# ResNet50 was pretrained on ImageNet — 1.2 million images.
# It already knows how to detect edges, textures, color
# gradients, and atmospheric patterns. Your 2-layer strided
# conv had to learn all of this from 12,000 images alone —
# an impossible task. ResNet50 gives CSPNet rich, meaningful
# features instead of shallow ones.
#
# WHAT STAYS THE SAME:
# Patchify function — identical to before
# CSPNet — identical to before
# Classification head — identical to before
# Training loop — identical to before
#
# Only the feature extractor before patchify changes.
# ============================================================

class ResNet50Backbone(nn.Module):

    def __init__(self):
        super().__init__()

        # ── LOAD PRETRAINED RESNET50 ──
        # weights='IMAGENET1K_V1' means load weights trained
        # on ImageNet. These weights encode 50 layers worth of
        # visual knowledge from 1.2 million images.
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # ── REMOVE LAST TWO LAYERS ──
        # ResNet50 ends with: avgpool → flatten → fc (classifier)
        # We don't want the classifier — we want the features.
        # list(resnet.children())[:-2] removes avgpool and fc.
        # What remains outputs shape: (batch, 2048, 7, 7)
        # 2048 channels of deeply learned features at 7x7 spatial
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # ── FREEZE BACKBONE WEIGHTS ──
        # WHY FREEZE? We don't want to change ResNet50's weights
        # during training. They are already excellent.
        # Freezing means only our new layers (reduce, upsample,
        # CSPNet, classifier) learn during training.
        # This also speeds up training significantly.
        for param in self.backbone.parameters():
            param.requires_grad = False

        # ── CHANNEL REDUCTION ──
        # ResNet50 outputs 2048 channels — too many for our
        # lightweight CSPNet. Reduce to 256 channels using
        # a 1x1 conv (channel mixing without spatial change).
        # WHY 256? Rich enough for CSPNet, light enough to train.
        # 1x1 conv: (batch, 2048, 7, 7) → (batch, 256, 7, 7)
        self.reduce = nn.Sequential(
            nn.Conv2d(2048, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        # ── UPSAMPLE TO 56x56 ──
        # WHY UPSAMPLE? Our patchify cuts 56x56 into
        # 4x4 = 16 patches of 14x14 each.
        # ResNet50 gives us 7x7 — too small for that.
        # Bilinear upsampling stretches 7x7 → 56x56
        # without any learnable parameters.
        # mode='bilinear': smooth interpolation between pixels
        # align_corners=False: standard setting for bilinear
        self.upsample = nn.Upsample(
            size=(56, 56),
            mode='bilinear',
            align_corners=False
        )

    def forward(self, x):
        # x: (batch, 3, 224, 224)

        x = self.backbone(x)   # → (batch, 2048, 7, 7)
        x = self.reduce(x)     # → (batch, 256, 7, 7)
        x = self.upsample(x)   # → (batch, 256, 56, 56)

        return x
        # Output: rich 256-channel feature map at 56x56
        # Ready for patchify — same as before


def patchify(feature_map, patch_size=14):
    """
    Cuts feature map into equal patches.
    IDENTICAL to before — nothing changed here.
    56x56 ÷ 14 = 4 patches per side = 16 patches total
    """
    batch_size, channels, height, width = feature_map.shape
    num_patches_h = height // patch_size
    num_patches_w = width  // patch_size
    patches = []

    for row in range(num_patches_h):
        for col in range(num_patches_w):
            r_start = row * patch_size
            c_start = col * patch_size
            patch   = feature_map[
                :, :,
                r_start:r_start+patch_size,
                c_start:c_start+patch_size
            ]
            patches.append(patch)

    return torch.stack(patches, dim=1)
    # Output: (batch, 16, 256, 14, 14)


class FeatureExtractorAndPatcher(nn.Module):
    """
    Chains ResNet50 backbone + patchify.
    Replaces the old FeatureExtractorAndPatcher_B.
    """

    def __init__(self, patch_size=14):
        super().__init__()
        self.backbone   = ResNet50Backbone()
        self.patch_size = patch_size

    def forward(self, x):
        # Step 1: ResNet50 extracts rich features
        # (batch, 3, 224, 224) → (batch, 256, 56, 56)
        x = self.backbone(x)

        # Step 2: Patchify — identical to before
        # (batch, 256, 56, 56) → (batch, 16, 256, 14, 14)
        patches = patchify(x, patch_size=self.patch_size)

        return patches


# ============================================================
# PHASE 4 — CSPNET + CLASSIFICATION HEAD
# IDENTICAL TO BEFORE — nothing changed here at all.
# Only in_channels changes: 128 → 256 to match backbone output
# ============================================================

class CSPBlock(nn.Module):
    """
    Cross Stage Partial Network block.
    IDENTICAL to before — no changes.
    Splits input, processes half, merges back.
    """

    def __init__(self, in_channels):
        super().__init__()
        half = in_channels // 2

        self.main_conv1 = nn.Conv2d(half, half, kernel_size=3, padding=1)
        self.main_bn1   = nn.BatchNorm2d(half)
        self.main_relu1 = nn.ReLU()

        self.main_conv2 = nn.Conv2d(half, half, kernel_size=3, padding=1)
        self.main_bn2   = nn.BatchNorm2d(half)
        self.main_relu2 = nn.ReLU()

        self.blend_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.blend_bn   = nn.BatchNorm2d(in_channels)
        self.blend_relu = nn.ReLU()

    def forward(self, x):
        main_input, skip_input = torch.chunk(x, chunks=2, dim=1)

        main = self.main_relu1(self.main_bn1(self.main_conv1(main_input)))
        main = self.main_relu2(self.main_bn2(self.main_conv2(main)))

        combined = torch.cat([main, skip_input], dim=1)
        out      = self.blend_relu(self.blend_bn(self.blend_conv(combined)))

        return out


class PatchCSPNet(nn.Module):
    """
    Processes all 16 patches through shared CSPBlock.
    Aggregates patch features by averaging.
    IDENTICAL to before — only in_channels changes 128→256
    """

    def __init__(self, in_channels=256):
        super().__init__()
        self.csp_block = CSPBlock(in_channels=in_channels)
        self.gap        = nn.AdaptiveAvgPool2d(output_size=1)

    def forward(self, patches):
        # patches: (batch, 16, 256, 14, 14)
        batch_size  = patches.shape[0]
        in_channels = patches.shape[2]  # 256
        patch_features = []

        for i in range(patches.shape[1]):  # loop 16 patches
            patch    = patches[:, i, :, :, :]           # (batch, 256, 14, 14)
            features = self.csp_block(patch)             # (batch, 256, 14, 14)
            features = self.gap(features)                # (batch, 256, 1, 1)
            features = features.view(batch_size, in_channels)  # (batch, 256)
            patch_features.append(features)

        all_features  = torch.stack(patch_features, dim=1)  # (batch, 16, 256)
        image_feature = all_features.mean(dim=1)             # (batch, 256)

        return image_feature


class ClassificationHead(nn.Module):
    """
    Maps 256-dim feature vector to 6 AQI class scores.
    IDENTICAL to before — only in_features changes 128→256
    """

    def __init__(self, in_features=256, num_classes=6):
        super().__init__()
        self.fc1     = nn.Linear(in_features, 512)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(p=0.4)
        self.fc2     = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class AQIClassifier(nn.Module):
    """
    Complete pipeline:
    Raw image → ResNet50 → Patchify → CSPNet → Classify
    """

    def __init__(self, num_classes=6, patch_size=14):
        super().__init__()

        # ResNet50 replaces StridedConvBlock
        self.feature_extractor = FeatureExtractorAndPatcher(
            patch_size=patch_size
        )
        # CSPNet — identical to before, just 256 channels
        self.patch_cspnet = PatchCSPNet(in_channels=256)

        # Classifier — identical to before, just 256 input
        self.classifier = ClassificationHead(
            in_features=256,
            num_classes=num_classes
        )

    def forward(self, x):
        patches        = self.feature_extractor(x)  # (batch, 16, 256, 14, 14)
        image_features = self.patch_cspnet(patches)  # (batch, 256)
        logits         = self.classifier(image_features)  # (batch, 6)
        return logits


# ============================================================
# TRAINING SETUP
#
# ONE IMPORTANT CHANGE FROM BEFORE:
# We now use TWO different learning rates:
#
# WHY TWO LEARNING RATES?
# ResNet50 backbone is frozen (no learning) — so it doesn't
# need a learning rate at all.
# But the reduce conv layer (1x1) is new and needs to learn.
# The CSPNet and classifier are also new and need to learn.
#
# All new layers use lr=0.0001
# This is standard practice called "fine-tuning"
# ============================================================

def setup_training(model):
    criterion = nn.CrossEntropyLoss()

    # Only pass parameters that require gradients
    # (frozen ResNet50 backbone params are excluded automatically)
    trainable_params = [p for p in model.parameters()
                        if p.requires_grad]

    optimizer = optim.Adam(trainable_params, lr=0.0001)

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )

    return criterion, optimizer, scheduler


def train_one_epoch(model, train_loader, criterion,
                    optimizer, device):
    model.train()
    total_loss, total_correct, total_images = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(images)
        loss        = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        predicted      = predictions.argmax(dim=1)
        total_correct += (predicted == labels).sum().item()
        total_images  += labels.size(0)

        if (batch_idx + 1) % 20 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)}"
                  f" | Loss: {loss.item():.4f}")

    return total_loss / len(train_loader), \
           (total_correct / total_images) * 100


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_images = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            predictions    = model(images)
            loss           = criterion(predictions, labels)

            total_loss    += loss.item()
            predicted      = predictions.argmax(dim=1)
            total_correct += (predicted == labels).sum().item()
            total_images  += labels.size(0)

    return total_loss / len(val_loader), \
           (total_correct / total_images) * 100


def full_training(model, train_loader, val_loader,
                  num_epochs=50, patience=10):

    device = torch.device("cuda" if torch.cuda.is_available()
                          else "cpu")
    print(f"Training on: {device}")
    model = model.to(device)

    criterion, optimizer, scheduler = setup_training(model)

    best_val_loss    = float('inf')
    patience_counter = 0
    best_model_path  = "best_aqi_model.pth"

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    print(f"\nStarting training for up to {num_epochs} epochs")
    print(f"Early stopping patience: {patience} epochs\n")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 40)

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device)

        val_loss, val_acc = validate(
            model, val_loader, criterion, device)

        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"  Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {val_loss:.4f} | "
              f"Val Acc:   {val_acc:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  ✅ Validation improved — model saved")
        else:
            patience_counter += 1
            print(f"  ⚠️  No improvement "
                  f"({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n🛑 Early stopping at epoch {epoch+1}")
                break

        print()

    model.load_state_dict(torch.load(best_model_path))
    print(f"✅ Best model loaded (val loss: {best_val_loss:.4f})")

    return model, train_losses, val_losses, train_accs, val_accs


# ============================================================
# VERIFY + TRAIN
# ============================================================

# Delete old checkpoint
if os.path.exists("best_aqi_model.pth"):
    os.remove("best_aqi_model.pth")
    print("Old checkpoint deleted")

# Build fresh model
model = AQIClassifier(num_classes=6, patch_size=14)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"\nModel built successfully")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}  (ResNet50 backbone)")

# Shape check
fake_batch = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    output = model(fake_batch)

print(f"\nShape check:")
print(f"  Input:    {list(fake_batch.shape)}")
print(f"  Output:   {list(output.shape)}")
print(f"  Expected: [4, 6]")

if list(output.shape) == [4, 6]:
    print(f"\n✅ Model verified. Starting training...")
else:
    print(f"\n❌ Shape mismatch — check architecture")

# Train
trained_model, train_losses, val_losses, \
train_accs, val_accs = full_training(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    num_epochs   = 50,
    patience     = 10
)